In [1]:
import os
import networkx as nx
import matplotlib.pyplot as plt
from netgraph import Graph
import time

In [2]:
def plot_community(G, comms):
    community_to_color = {
        0 : 'tab:blue',
        1 : 'tab:orange',
        2 : 'tab:green',
        3 : 'tab:red',
        4 : 'tab:cyan',
        5 : 'tab:pink',
        6 : 'tab:purple',
    }
    node_color = {node: community_to_color[community_id] for node, community_id in comms.items()}

    Graph(G,
        node_color=node_color, node_edge_width=0.1, edge_alpha=0.5,
        node_layout='community', node_layout_kwargs=dict(node_to_community=comms),
        #   edge_layout='bundled', edge_layout_kwargs=dict(k=2000),
    )
    plt.show()

In [3]:
seed = 42

# Number of nodes
n = 2394385

# Degree distribution
t1 = 2.0733548989807504
d_min = 1
d_mean = 2.760357487264965
d_max = 31
d_max_iter = 1000

# Community size distribution
t2 = 1.8739001030982472
c_min = 8
c_max = 4658
c_max_iter = 1000

# **LFR**

In [4]:
mu = 0.35486504665095914

In [5]:
start = time.perf_counter()
os.system(f'./package1/binary_networks/benchmark -t1 {t1} -t2 {t2} -N {n} -k {d_mean} -maxk {d_max} -mu {mu} -minc {c_min} -maxc {c_max}')
elapsed = time.perf_counter() - start
print(f"Generation time: {elapsed}")

setting... -t1 2.0733548989807504
setting... -t2 1.8739001030982472
setting... -N 2394385
setting... -k 2.760357487264965
setting... -maxk 31
setting... -mu 0.35486504665095914
setting... -minc 8
setting... -maxc 4658

**************************************************************
number of nodes:	2394385
average degree:	2.76036
maximum degree:	31
exponent for the degree distribution:	2.07335
exponent for the community size distribution:	1.8739
mixing parameter:	0.354865
number of overlapping nodes:	0
number of memberships of the overlapping nodes:	0
community size range set equal to [8 , 4658]
**************************************************************

Generation time: 0.0024200690004363423



***********************
ERROR: the average degree is out of range:
you should increase the average degree (bigger than 3.3421)
(or decrease the maximum degree...)


In [ ]:
G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open('network.dat').readlines()], nodetype=int)

In [ ]:
# comms = {}
# for line in open('community.dat').readlines():
#     node, comm = map(int, line.strip().split('\t'))
#     comms[node] = comm - 1
    
# plot_community(G, comms)

In [ ]:
degree = []
for u in G.nodes:
    degree.append(len(G[u]))
degree = sorted(degree, reverse=True)
    
with open('deg.dat', 'w') as f:
    f.write('\n'.join(map(str, degree)))


In [ ]:
cs = {}
for line in open('community.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    cs.setdefault(comm, 0)
    cs[comm] += 1
cs = sorted(cs.values(), reverse=True)

with open('cs.dat', 'w') as f:
    f.write('\n'.join(map(str, cs)))

# **ABCD**

In [ ]:
xi = mu

In [ ]:
start = time.perf_counter()
os.system(f'julia ABCDGraphGenerator.jl/utils/graph_sampler.jl edge.dat com.dat deg.dat cs.dat xi {xi} false false {seed} 0')
elapsed = time.perf_counter() - start
print(f"Generation time: {elapsed}")

In [ ]:
G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open('edge.dat').readlines()], nodetype=int)

In [ ]:
comms = {}
for line in open('com.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    comms[node] = comm - 1
    
plot_community(G, comms)

In [ ]:
with open('abcd_config.toml', 'w') as f:
    f.write(
        f'''seed = "{seed}"                   # RNG seed, use "" for no seeding
n = "{n}"                   # number of vertices in graph
t1 = "{t1}"                      # power-law exponent for degree distribution
d_min = "{d_min}"                   # minimum degree
d_max = "{d_max}"                  # maximum degree
d_max_iter = "{d_max_iter}"           # maximum number of iterations for sampling degrees
t2 = "{t2}"                      # power-law exponent for cluster size distribution
c_min = "{c_min}"                  # minimum cluster size
c_max = "{c_max}"                # maximum cluster size
c_max_iter = "{c_max_iter}"           # maximum number of iterations for sampling cluster sizes
# Exactly one of xi and mu must be passed as Float64. Also if xi is provided islocal must be set to false or omitted.
xi = "{xi}"                    # fraction of edges to fall in background graph
#mu = "0.2"                   # mixing parameter
islocal = "false"             # if "true" mixing parameter is restricted to local cluster, otherwise it is global
isCL = "false"                # if "false" use configuration model, if "true" use Chung-Lu
degreefile = "deg.dat"        # name of file do generate that contains vertex degrees
communitysizesfile = "cs.dat" # name of file do generate that contains community sizes
communityfile = "com.dat"     # name of file do generate that contains assignments of vertices to communities
networkfile = "edge.dat"      # name of file do generate that contains edges of the generated graph
nout = "0"                  # number of vertices in graph that are outliers; optional parameter
                            # if nout is passed and is not zero then we require islocal = "false",
                            # isCL = "false", and xi (not mu) must be passed
                            # if nout > 0 then it is recommended that xi > 0'''        
    )

In [ ]:
os.system('julia ABCDGraphGenerator.jl/utils/abcd_sampler.jl abcd_config.toml')

G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open('edge.dat').readlines()], nodetype=int)

comms = {}
for line in open('com.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    comms[node] = comm - 1
    
plot_community(G, comms)